<a href="https://colab.research.google.com/github/monaa96/Watercooler-talk/blob/main/Copy_of_Fine_Tuning_HODL_Final_Project_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Water-Cooler-Talk




## Introduction

In [ ]:
import os
os.environ['KERAS_BACKEND'] = 'tensorflow'

In [ ]:
import keras
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler

In [ ]:
keras.utils.set_random_seed(42)

## Retrieving and preparing the Data


In [ ]:
soccer_path = "https://drive.google.com/uc?export=download&id=172TZqufIgdEGIM2aNz-TnjmnuN5lxhz-"
cricket_path = "https://drive.google.com/uc?export=download&id=1mYSEp7IxEQ4uNAyFPYU0DrjBWA6UNWMz"
FPL_path = "https://drive.google.com/file/d/1F0Rsz0DggoNq2tFkyG4xfXDU50WIXcxQ/view?usp=drive_link"


In [ ]:
# Read data from URL

soccer_full_df = pd.read_csv(soccer_path)
cricket_full_df = pd.read_csv(cricket_path)



print(f"""
soccer samples: {soccer_full_df.shape[0]}
cricket samples: {cricket_full_df.shape[0]}
""")


soccer samples: 8889
cricket samples: 66317



# Data Cleanup

In [ ]:
soccer_full_df.head()

,title,score,id,url,comms_num,created,body,timestamp
0,Best Goalkeeper Saves in Euro 2020 | HD #1,1,o7n9m2,https://youtu.be/YUF1js6hnXc,1,1.624655e+09,NaN,2021-06-26 00:02:47
1,ITALY EURO UEFA 2020 (2021),3,o7mz20,https://youtu.be/B7oXcH3YDdo,0,1.624654e+09,NaN,2021-06-25 23:44:58
2,PORTUGAL UEFA EURO 2020 (2021),1,o7m096,https://youtu.be/ImjAX6ruazQ,0,1.624650e+09,NaN,2021-06-25 22:41:51
3,Italy v Austria - A Tactical Preview,1,o7lde2,https://twitter.com/pythaginboots/status/14083...,0,1.624647e+09,NaN,2021-06-25 21:57:28
4,Euro 2020: Players To Buy For The Upcoming Mat...,1,o7jzus,https://justonside.com/2021/06/25/euro-2020-kn...,0,1.624641e+09,NaN,2021-06-25 20:05:57


In [ ]:
def normalize_column(df, column_name):
  """Normalizes a column of data in a Pandas DataFrame using MinMaxScaler.

  Args:
    df: The Pandas DataFrame.
    column_name: The name of the column to normalize.

  Returns:
    The DataFrame with the specified column normalized.
  """
  # Create a MinMaxScaler object
  scaler = MinMaxScaler()

  # Reshape the column values before scaling
  values = df[column_name].values.reshape(-1, 1)

  # Fit the scaler to the column data and transform it
  normalized_values = scaler.fit_transform(values)

  # Update the column with the normalized values
  df[column_name] = normalized_values

  # Update the column name of normalized value to rating
  df = df.rename(columns={column_name: 'rating'})

  return df

In [ ]:
# Data cleanup for soccer

print(soccer_full_df.columns)
soccer_full_df_sample = normalize_column(soccer_full_df, 'score')
# Select rows where 'title' is equal to 'comment'
filtered_soccer_data = soccer_full_df_sample[soccer_full_df_sample['title'] == 'Comment']

#now keep only the comments, and the score (how much people liked or disliked the comment)
filtered_soccer_data = filtered_soccer_data[['body', 'rating']]

# Display the selected data
filtered_soccer_data.head()


Index(['title', 'score', 'id', 'url', 'comms_num', 'created', 'body',
       'timestamp'],
      dtype='object')


,body,rating
15,Selling two quarter final tickets in Rome. Not...,0.020942
16,Think we (denmark) can take them in the quarte...,0.020194
17,"It really doesn’t, pay very close attention it...",0.020194
18,"Now I'm biased, but wouldn't you say Denmark h...",0.021690
19,Rewatch the goal. It gets deflected by at leas...,0.020194


In [ ]:
cricket_full_df.head()

,Unnamed: 0,id,body,author,upvotes,timestamp
0,0,k9wfxsr,"Now, everyone watching in Australia, go the fu...",Grolschisgood,403.0,2023-11-19 15:55:22
1,1,k9ve9vv,My grandpa got traumatized by Aussies in 1987 ...,creatorop,198.0,2023-11-19 09:30:06
2,2,k9wfw0u,One of the most strongly-attended funerals I'v...,NotWearingNails,151.0,2023-11-19 15:55:01
3,3,k9wfxzf,Travis Head has just put himself on a list con...,brbr0433,433.0,2023-11-19 15:55:25
4,4,k9v7vet,"Why did Cummins have to do a classic ""We'll ha...",DazBoy11,414.0,2023-11-19 08:04:41


In [ ]:
# Data cleanup for cricket

print(cricket_full_df.columns)
cricket_full_df_sample = normalize_column(cricket_full_df, 'upvotes')
#now keep only the comments, and the score (how much people liked or disliked the comment)
filtered_cricket_data = cricket_full_df_sample[['body', 'rating']]

# Display the selected data
filtered_cricket_data.head()

Index(['Unnamed: 0', 'id', 'body', 'author', 'upvotes', 'timestamp'], dtype='object')


,body,rating
0,"Now, everyone watching in Australia, go the fu...",0.448347
1,My grandpa got traumatized by Aussies in 1987 ...,0.236570
2,One of the most strongly-attended funerals I'v...,0.188017
3,Travis Head has just put himself on a list con...,0.479339
4,"Why did Cummins have to do a classic ""We'll ha...",0.459711


# Now we will combine the datasets and train the model

Cleaning dataset for LLM

In [ ]:
Final_df = pd.concat([filtered_cricket_data[:500], filtered_soccer_data[:500]], axis=0)
#note I am selecting the first 500 comments of each here, to help with training speed later

Final_df.head()

##Convert to lower case
Final_df['body'] = Final_df['body'].str.lower()

print(Final_df['body'])

train_texts = Final_df['body']
print(len(train_texts))


0      now, everyone watching in australia, go the fu...
1      my grandpa got traumatized by aussies in 1987 ...
2      one of the most strongly-attended funerals i'v...
3      travis head has just put himself on a list con...
4      why did cummins have to do a classic "we'll ha...
                             ...                        
564    lolwot? this isn't what happened. the way you'...
565                                     perfect bromance
566    hindsight is 20-20. let’s be honest. a week ag...
567    it seems you have / had no idea that finishing...
568                                           that hurts
Name: body, Length: 1000, dtype: object
1000


Importing a pre-trained model

In [ ]:
pip install transformers #using transformers not Keras Hub

In [ ]:
from transformers import BartForConditionalGeneration, BartTokenizer

In [ ]:

#importing a pre-trained model - BART - which is better for text generation and summarization
model_name = "facebook/bart-large-cnn"
tokenizer = BartTokenizer.from_pretrained(model_name, model_max_length=1024, truncation=True)
model = BartForConditionalGeneration.from_pretrained(model_name)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.58k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

Vetor and Tokenize

In [ ]:
print(model.config)

BartConfig {
  "_attn_implementation_autoset": true,
  "_name_or_path": "facebook/bart-large-cnn",
  "_num_labels": 3,
  "activation_dropout": 0.0,
  "activation_function": "gelu",
  "add_final_layer_norm": false,
  "architectures": [
    "BartForConditionalGeneration"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 0,
  "classif_dropout": 0.0,
  "classifier_dropout": 0.0,
  "d_model": 1024,
  "decoder_attention_heads": 16,
  "decoder_ffn_dim": 4096,
  "decoder_layerdrop": 0.0,
  "decoder_layers": 12,
  "decoder_start_token_id": 2,
  "dropout": 0.1,
  "early_stopping": true,
  "encoder_attention_heads": 16,
  "encoder_ffn_dim": 4096,
  "encoder_layerdrop": 0.0,
  "encoder_layers": 12,
  "eos_token_id": 2,
  "force_bos_token_to_be_generated": true,
  "forced_bos_token_id": 0,
  "forced_eos_token_id": 2,
  "gradient_checkpointing": false,
  "id2label": {
    "0": "LABEL_0",
    "1": "LABEL_1",
    "2": "LABEL_2"
  },
  "init_std": 0.02,
  "is_encoder_decoder": true,
  "label2id": {
  

In [ ]:
# Tokenize the training and validation texts
train_encodings = tokenizer(list(train_texts), truncation=True, padding=True)

:Prepare Data for Training

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

class CommentDataset(Dataset):
    def __init__(self, encodings):
        self.encodings = encodings

    def __getitem__(self, idx):
        return {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}

    def __len__(self):
        return len(self.encodings.input_ids)

train_dataset = CommentDataset(train_encodings)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True) #note was running into memory issues with larger batch sizes

Fine-tuning

In [ ]:
!pip install tqdm
from tqdm import tqdm #importing the tqdm module for the progress bar

from transformers import AdamW

optimizer = AdamW(model.parameters(), lr=2e-4)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print(f"Using device: {device}")  # Print connection status (for troubleshooting)


num_epochs = 4

for epoch in range(num_epochs):
    model.train()

    # Wrap the training loop with tqdm for a progress bar
    with tqdm(train_loader, unit="batch") as tepoch:
        for batch in tepoch:
            tepoch.set_description(f"Epoch {epoch + 1}")

            optimizer.zero_grad()
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["input_ids"].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            loss.backward()
            optimizer.step()

            tepoch.set_postfix(loss=loss.item())


# Save the fine-tuned model
model.save_pretrained("fine_tuned_bart")
tokenizer.save_pretrained("fine_tuned_bart")

/usr/local/lib/python3.11/dist-packages/transformers/optimization.py:591: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


Using device: cuda


Epoch 4: 100%|██████████| 125/125 [01:31<00:00,  1.36batch/s, loss=0.00208]
/usr/local/lib/python3.11/dist-packages/transformers/modeling_utils.py:2758: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 142, 'min_length': 56, 'early_stopping': True, 'num_beams': 4, 'length_penalty': 2.0, 'no_repeat_ngram_size': 3, 'forced_bos_token_id': 0}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


('fine_tuned_bart/tokenizer_config.json',
 'fine_tuned_bart/special_tokens_map.json',
 'fine_tuned_bart/vocab.json',
 'fine_tuned_bart/merges.txt',
 'fine_tuned_bart/added_tokens.json')

Comparing original model to fine tuned

In [ ]:
# Load the original BART model
model_og = BartForConditionalGeneration.from_pretrained("facebook/bart-large-cnn")

# Load your fine-tuned BART model
model_ft = BartForConditionalGeneration.from_pretrained("fine_tuned_bart")

original_weights = model_og.state_dict()
fine_tuned_weights = model_ft.state_dict()

# Compare weights between models
total_diff = 0
num_weights = 0
total_weights = 0

for key in original_weights:
    if key in fine_tuned_weights:
        diff = torch.abs(original_weights[key] - fine_tuned_weights[key]).mean().item()
        total_diff += diff
        num_weights += 1
        total_weights += torch.abs(original_weights[key]).sum().item()

avg_diff = total_diff / num_weights
print(f"Average weight difference: {avg_diff}")

avg_weight = total_weights / num_weights
print(f"Average weight: {avg_weight}")


/usr/local/lib/python3.11/dist-packages/transformers/models/bart/configuration_bart.py:176: UserWarning: Please make sure the config includes `forced_bos_token_id=0` in future versions. The config can simply be saved and uploaded again to be fixed.
  warnings.warn(


Average weight difference: 0.0010978943623426787
Average weight: 56089.34588439268


Testing the model

In [ ]:
print(model_ft.config)  # Print model configuration


BartConfig {
  "_attn_implementation_autoset": true,
  "_name_or_path": "fine_tuned_bart",
  "_num_labels": 3,
  "activation_dropout": 0.0,
  "activation_function": "gelu",
  "add_final_layer_norm": false,
  "architectures": [
    "BartForConditionalGeneration"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 0,
  "classif_dropout": 0.0,
  "classifier_dropout": 0.0,
  "d_model": 1024,
  "decoder_attention_heads": 16,
  "decoder_ffn_dim": 4096,
  "decoder_layerdrop": 0.0,
  "decoder_layers": 12,
  "decoder_start_token_id": 2,
  "dropout": 0.1,
  "early_stopping": null,
  "encoder_attention_heads": 16,
  "encoder_ffn_dim": 4096,
  "encoder_layerdrop": 0.0,
  "encoder_layers": 12,
  "eos_token_id": 2,
  "force_bos_token_to_be_generated": true,
  "forced_bos_token_id": 0,
  "forced_eos_token_id": 2,
  "gradient_checkpointing": false,
  "id2label": {
    "0": "LABEL_0",
    "1": "LABEL_1",
    "2": "LABEL_2"
  },
  "init_std": 0.02,
  "is_encoder_decoder": true,
  "label2id": {
    "LABEL

In [ ]:
#EXAMPLE ARTICLE / input
text_to_summarize = "Manchester United skipper Bruno Fernandes scored a hat-trick to lead his side to victory against 10-man Real Sociedad at Old Trafford and book a Europa League quarter-final encounter with French club Lyon. Needing a victory to keep alive their hopes of marking Ruben Amorims first year in charge with silverware and knowing the tournament is their only realistic chance of securing European football next season, United got off to the worst possible start as Mikel Oyarzabal scored from the spot after being brought down by Matthijs de Ligt. Thankfully for the hosts, the were level within six minutes as Fernandes kept his cool after Igor Zubeldia had taken Rasmus Hojlund down as the Dane was bearing down on the Sociedad goal. Remarkably, it took a third penalty of the night to get Uniteds noses in front. Aritz Elustondo complained long after it was ruled he had fouled Patrick Dorgu. It made no difference, and for the third time, the goalkeeper went the wrong way as Fernandes found the net for from the spot for his 14th goal of the season. The Spanish visitors last chance seemed to disappear midway through the second half when Jon Aramburu was sent off for hauling down Dorgu, who would have been through. But just for good measure, United struck twice more before the end.Fernandes completed his second hat-trick for the club with a low finish across goal and into the far corner before fellow Portugal international Diogo Dalot drilled home a fourth in stoppage time."

In [ ]:
model_name_ft = "fine_tuned_bart"  # name of fine tuned model
model_ft = BartForConditionalGeneration.from_pretrained(model_name_ft, force_download=True)
tokenizer_ft = BartTokenizer.from_pretrained(model_name_ft, model_max_length=1024, truncation=True, force_download=True)

In [ ]:
model_name_og = "facebook/bart-large-cnn"
tokenizer_og = BartTokenizer.from_pretrained(model_name_og, model_max_length=1024, truncation=True)
model_og = BartForConditionalGeneration.from_pretrained(model_name_og)

In [ ]:
# Sending the input to the fine tuned model
inputs = tokenizer_ft.batch_encode_plus([text_to_summarize],return_tensors='pt')
summary_ids =  model_ft.generate(inputs['input_ids'], num_beams=15, max_length=150, early_stopping=True, no_repeat_ngram_size = 2)

/usr/local/lib/python3.11/dist-packages/transformers/generation/utils.py:1528: UserWarning: You have modified the pretrained model configuration to control generation. This is a deprecated strategy to control generation and will be removed in v5. Please use and modify the model generation configuration (see https://huggingface.co/docs/transformers/generation_strategies#default-text-generation-configuration )
  warnings.warn(


In [ ]:
# Reading the summary for the fine tuned model
summary = tokenizer_ft.decode(summary_ids[0], skip_special_tokens=True)

if summary.strip() == "":  # Check if summary is empty or contains only spaces - since this was a problem occuring
    print("Model generated a blank/space output.")
else:
    print(summary)


Manchester United skipper Bruno Fernandes scored a hat-trick to lead his side to victory against 10-man Real Sociedad at Old Trafford and book a Europa League quarter-final encounter with French club Lyon. Needing a victory to keep alive their hopes of marking Ruben Amorims first year in charge with silverware and knowing the tournament is their only realistic chance of securing European football next season, United got off to the worst possible start as Mikel Oyarzabal scored from the spot after being brought down by Matthijs de Ligt. Thankfully for the hosts, the were level within six minutes. The Spanish visitors last chance seemed to disappear midway through the second half when Jon Aramburu was


In [ ]:
#Comparison to regular model - sending inputs here
inputs = tokenizer_og.batch_encode_plus([text_to_summarize],return_tensors='pt')
summary_ids =  model_og.generate(inputs['input_ids'], num_beams=4, max_length=150, early_stopping=True)

In [ ]:
# Decoding and printing the summary
summary = tokenizer_og.decode(summary_ids[0], skip_special_tokens=True)
print(summary)

Manchester United beat Real Sociedad 4-1 at Old Trafford. Mikel Oyarzabal, Bruno Fernandes and Diogo Dalot scored the goals. Jon Aramburu was sent off for the visitors in the second half. United will now face Lyon in the quarter-finals.
